# 02 — Shared Development / Holdout Split and Cross-Validation Registry

## Purpose

This notebook creates the single shared data-splitting registry that will be used by all six machine-learning models in the project.

It will:

- verify that the correct dataset is being used;
- create a stratified 80% development / 20% formal holdout split;
- create five stratified cross-validation folds within the development partition;
- use `random_state=42` for reproducibility;
- preserve `Employee_ID` only for record tracking;
- save the final split and fold assignments for use by every group member.

### Important rules

- The formal holdout must not be used for preprocessing decisions, hyperparameter tuning, early stopping, threshold selection, calibration selection, or model selection.
- All six models must use the same saved development/holdout partition and the same five cross-validation folds.
- Individual members must not create separate train/test splits.
- Model-specific preprocessing will be fitted later using training data only.

### Models using this registry

1. FT-Transformer
2. CatBoost Classifier
3. XGBoost Classifier
4. LightGBM Classifier
5. Explainable Boosting Machine (EBM)
6. TabNet

In [1]:
from pathlib import Path
import hashlib

import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold


# The notebook is inside the notebooks/ folder,
# so ".." points to the project root.
project_root = Path("..").resolve()

# Raw dataset
data_file = (
    project_root
    / "data"
    / "raw"
    / "Extended_Employee_Performance_and_Productivity_Data.csv"
)

# Folder where the shared split registry will be saved
processed_folder = project_root / "data" / "processed"
processed_folder.mkdir(parents=True, exist_ok=True)

# Final registry output path
split_registry_file = processed_folder / "split_registry.csv"

print("Project root:", project_root)
print("Dataset:", data_file)
print("Split registry output:", split_registry_file)

Project root: D:\Desktop\Employee_Resignation_Prediction
Dataset: D:\Desktop\Employee_Resignation_Prediction\data\raw\Extended_Employee_Performance_and_Productivity_Data.csv
Split registry output: D:\Desktop\Employee_Resignation_Prediction\data\processed\split_registry.csv


In [2]:
# Expected SHA-256 checksum of the exact dataset approved for this project.
EXPECTED_SHA256 = (
    "2b1e07caf58e7678a780d428c1a8893a"
    "95d9c4a556134fd8e7368db5ed596cdb"
)


# Check that the dataset actually exists at the expected location.
if not data_file.exists():
    raise FileNotFoundError(
        f"Dataset not found at:\n{data_file}"
    )


# Calculate the SHA-256 checksum of the dataset.
# Reading the file in chunks avoids loading the whole file into memory at once.
sha256 = hashlib.sha256()

with open(data_file, "rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()


# Display the checksum so it is recorded in the notebook output.
print("Dataset found:", data_file)
print("Expected SHA-256:", EXPECTED_SHA256)
print("Actual SHA-256:  ", actual_sha256)


# Stop the notebook immediately if this is not the exact approved dataset.
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError(
        "Dataset checksum does not match the approved project dataset."
    )

print("\nDataset checksum verification: PASSED")

Dataset found: D:\Desktop\Employee_Resignation_Prediction\data\raw\Extended_Employee_Performance_and_Productivity_Data.csv
Expected SHA-256: 2b1e07caf58e7678a780d428c1a8893a95d9c4a556134fd8e7368db5ed596cdb
Actual SHA-256:   2b1e07caf58e7678a780d428c1a8893a95d9c4a556134fd8e7368db5ed596cdb

Dataset checksum verification: PASSED


In [3]:
# Load the verified raw dataset into a pandas DataFrame.
df = pd.read_csv(data_file)


# Expected raw dataset structure for this project.
EXPECTED_ROWS = 100_000
EXPECTED_COLUMNS = 20


# Display the basic dataset dimensions.
print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


# Stop the notebook if the dataset dimensions are not what we expect.
if df.shape[0] != EXPECTED_ROWS:
    raise ValueError(
        f"Unexpected row count. "
        f"Expected {EXPECTED_ROWS:,}, found {df.shape[0]:,}."
    )

if df.shape[1] != EXPECTED_COLUMNS:
    raise ValueError(
        f"Unexpected column count. "
        f"Expected {EXPECTED_COLUMNS}, found {df.shape[1]}."
    )


print("\nDataset shape verification: PASSED")

Dataset loaded successfully.
Rows: 100000
Columns: 20

Dataset shape verification: PASSED


In [4]:
# Define the exact column order expected in the approved raw dataset.
EXPECTED_COLUMN_ORDER = [
    "Employee_ID",
    "Department",
    "Gender",
    "Age",
    "Job_Title",
    "Hire_Date",
    "Years_At_Company",
    "Education_Level",
    "Performance_Score",
    "Monthly_Salary",
    "Work_Hours_Per_Week",
    "Projects_Handled",
    "Overtime_Hours",
    "Sick_Days",
    "Remote_Work_Frequency",
    "Team_Size",
    "Training_Hours",
    "Promotions",
    "Employee_Satisfaction_Score",
    "Resigned",
]


# Check that the columns match exactly, including their order.
actual_columns = df.columns.tolist()

if actual_columns != EXPECTED_COLUMN_ORDER:
    raise ValueError(
        "Dataset columns do not match the approved project schema."
    )


# Employee_ID will be used only for record tracking.
# It must therefore be complete and unique.
missing_employee_ids = df["Employee_ID"].isna().sum()
duplicate_employee_ids = df["Employee_ID"].duplicated().sum()

if missing_employee_ids != 0:
    raise ValueError(
        f"Employee_ID contains {missing_employee_ids} missing values."
    )

if duplicate_employee_ids != 0:
    raise ValueError(
        f"Employee_ID contains {duplicate_employee_ids} duplicate values."
    )


# Check the target column.
# The raw dataset should contain only True and False.
target_values = set(df["Resigned"].dropna().unique())

if target_values != {True, False}:
    raise ValueError(
        f"Unexpected target values found: {target_values}"
    )


# Display a short validation summary.
print("Column schema: PASSED")
print("Missing Employee_ID values:", missing_employee_ids)
print("Duplicate Employee_ID values:", duplicate_employee_ids)
print("Target values:", sorted(target_values))
print("\nDataset identity and schema verification: PASSED")

Column schema: PASSED
Missing Employee_ID values: 0
Duplicate Employee_ID values: 0
Target values: [np.False_, np.True_]

Dataset identity and schema verification: PASSED


In [5]:
# Count how many employees stayed and how many resigned.
target_counts = df["Resigned"].value_counts().sort_index()

# Convert the counts into percentages.
target_percentages = (
    df["Resigned"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)


# Display the target distribution.
print("TARGET DISTRIBUTION")
print("-" * 35)

print(f"Stayed   (False): {target_counts[False]:,}")
print(f"Resigned (True):  {target_counts[True]:,}")

print()
print(f"Stayed percentage:   {target_percentages[False]:.2f}%")
print(f"Resigned percentage: {target_percentages[True]:.2f}%")


# Verify the expected total number of records.
if target_counts.sum() != len(df):
    raise ValueError(
        "Target counts do not add up to the total number of dataset rows."
    )


# Verify that both classes are present.
if len(target_counts) != 2:
    raise ValueError(
        "The target must contain exactly two classes."
    )


print("\nTarget distribution verification: PASSED")

TARGET DISTRIBUTION
-----------------------------------
Stayed   (False): 89,990
Resigned (True):  10,010

Stayed percentage:   89.99%
Resigned percentage: 10.01%

Target distribution verification: PASSED


In [6]:
# ------------------------------------------------------------
# CREATE THE OFFICIAL DEVELOPMENT / HOLDOUT SPLIT
# ------------------------------------------------------------

# Fixed project settings.
# These values must remain the same for every group member.
HOLDOUT_SIZE = 0.20
RANDOM_STATE = 42


# We split Employee_ID values rather than creating separate copies
# of the full dataset. Employee_ID is used only for record tracking.
employee_ids = df["Employee_ID"]

# The target is used only for stratification.
# Stratification keeps approximately the same resignation ratio
# in both the development and holdout partitions.
target = df["Resigned"]


# Create the official 80% development / 20% holdout split.
development_ids, holdout_ids = train_test_split(
    employee_ids,
    test_size=HOLDOUT_SIZE,
    random_state=RANDOM_STATE,
    stratify=target,
)


# Convert the returned values to sets for easy membership checking later.
development_ids = set(development_ids)
holdout_ids = set(holdout_ids)


# Display the partition sizes.
print("OFFICIAL DATA SPLIT")
print("-" * 35)

print(f"Total records:       {len(df):,}")
print(f"Development records: {len(development_ids):,}")
print(f"Holdout records:     {len(holdout_ids):,}")


# Verify that no employee appears in both partitions.
overlap = development_ids.intersection(holdout_ids)

if len(overlap) != 0:
    raise ValueError(
        f"{len(overlap)} Employee_ID values appear in both partitions."
    )


# Verify that every employee has been assigned exactly once.
assigned_ids = development_ids.union(holdout_ids)

if len(assigned_ids) != len(df):
    raise ValueError(
        "Not every employee was assigned to exactly one partition."
    )


print("\nDevelopment / holdout split verification: PASSED")

OFFICIAL DATA SPLIT


-----------------------------------
Total records:       100,000
Development records: 80,000
Holdout records:     20,000

Development / holdout split verification: PASSED


In [7]:
# ------------------------------------------------------------
# VERIFY TARGET DISTRIBUTION IN BOTH PARTITIONS
# ------------------------------------------------------------

# Create Boolean masks using the saved Employee_ID assignments.
development_mask = df["Employee_ID"].isin(development_ids)
holdout_mask = df["Employee_ID"].isin(holdout_ids)


# Calculate resignation prevalence for the full dataset.
full_resignation_rate = df["Resigned"].mean() * 100

# Calculate resignation prevalence for the development partition.
development_resignation_rate = (
    df.loc[development_mask, "Resigned"].mean() * 100
)

# Calculate resignation prevalence for the holdout partition.
holdout_resignation_rate = (
    df.loc[holdout_mask, "Resigned"].mean() * 100
)


# Count positive and negative cases in each partition.
development_counts = (
    df.loc[development_mask, "Resigned"]
    .value_counts()
    .sort_index()
)

holdout_counts = (
    df.loc[holdout_mask, "Resigned"]
    .value_counts()
    .sort_index()
)


# Display the results.
print("STRATIFICATION CHECK")
print("-" * 40)

print(f"Full dataset resignation rate: {full_resignation_rate:.2f}%")
print()

print("Development partition:")
print(f"  Stayed:   {development_counts[False]:,}")
print(f"  Resigned: {development_counts[True]:,}")
print(f"  Resignation rate: {development_resignation_rate:.2f}%")
print()

print("Holdout partition:")
print(f"  Stayed:   {holdout_counts[False]:,}")
print(f"  Resigned: {holdout_counts[True]:,}")
print(f"  Resignation rate: {holdout_resignation_rate:.2f}%")


# The two partition rates should be extremely close to the
# overall dataset resignation rate because stratification was used.
MAX_ALLOWED_DIFFERENCE = 0.05  # percentage points

if abs(development_resignation_rate - full_resignation_rate) > MAX_ALLOWED_DIFFERENCE:
    raise ValueError(
        "Development partition resignation rate differs too much "
        "from the full dataset."
    )

if abs(holdout_resignation_rate - full_resignation_rate) > MAX_ALLOWED_DIFFERENCE:
    raise ValueError(
        "Holdout partition resignation rate differs too much "
        "from the full dataset."
    )


print("\nStratification verification: PASSED")

STRATIFICATION CHECK
----------------------------------------
Full dataset resignation rate: 10.01%

Development partition:
  Stayed:   71,992
  Resigned: 8,008
  Resignation rate: 10.01%

Holdout partition:
  Stayed:   17,998
  Resigned: 2,002
  Resignation rate: 10.01%

Stratification verification: PASSED


In [8]:
# ------------------------------------------------------------
# CREATE THE FIVE SHARED CROSS-VALIDATION FOLDS
# ------------------------------------------------------------

# Keep only the development records.
# The formal holdout must not participate in cross-validation.
development_df = df.loc[development_mask].copy()


# Create the 5-fold stratified cross-validation object.
# shuffle=True randomizes the row assignment before creating folds.
# random_state=42 makes the fold assignments reproducible.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


# Create a column that will store each development employee's fold number.
# We initialize it as missing and fill it during the loop below.
development_df["cv_fold"] = pd.NA


# StratifiedKFold returns:
# - training indices
# - validation indices
#
# For the registry, we only need to save which validation fold
# each employee belongs to.
for fold_number, (_, validation_indices) in enumerate(
    cv.split(
        development_df,
        development_df["Resigned"],
    ),
    start=1,
):
    development_df.iloc[
        validation_indices,
        development_df.columns.get_loc("cv_fold")
    ] = fold_number


# Convert the fold column to an integer type.
development_df["cv_fold"] = development_df["cv_fold"].astype("int64")


# Display the number of employees assigned to each fold.
fold_counts = development_df["cv_fold"].value_counts().sort_index()

print("CROSS-VALIDATION FOLD ASSIGNMENTS")
print("-" * 40)

for fold_number, count in fold_counts.items():
    print(f"Fold {fold_number}: {count:,} records")


# Verify that exactly five folds were created.
if set(development_df["cv_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise ValueError(
        "The development data was not assigned to exactly five folds."
    )


# Verify that every development record has a fold assignment.
missing_fold_assignments = development_df["cv_fold"].isna().sum()

if missing_fold_assignments != 0:
    raise ValueError(
        f"{missing_fold_assignments} development records "
        "do not have a CV fold assignment."
    )


# Verify that all 80,000 development records were assigned.
if len(development_df) != len(development_ids):
    raise ValueError(
        "Development row count does not match the official split."
    )


print("\nFive-fold cross-validation assignment: PASSED")

CROSS-VALIDATION FOLD ASSIGNMENTS
----------------------------------------
Fold 1: 16,000 records
Fold 2: 16,000 records
Fold 3: 16,000 records
Fold 4: 16,000 records
Fold 5: 16,000 records

Five-fold cross-validation assignment: PASSED


In [9]:
# ------------------------------------------------------------
# VERIFY TARGET DISTRIBUTION INSIDE EACH CV FOLD
# ------------------------------------------------------------

# Calculate the overall resignation rate inside the
# development partition.
development_rate = (
    development_df["Resigned"].mean() * 100
)


print("CV FOLD STRATIFICATION CHECK")
print("-" * 45)

print(
    f"Development resignation rate: "
    f"{development_rate:.2f}%"
)
print()


# Check each fold separately.
for fold_number in range(1, 6):

    # Select only the records assigned to this fold.
    fold_data = development_df[
        development_df["cv_fold"] == fold_number
    ]

    # Count stayed and resigned employees.
    fold_counts = (
        fold_data["Resigned"]
        .value_counts()
        .sort_index()
    )

    # Calculate the resignation percentage.
    fold_rate = (
        fold_data["Resigned"].mean() * 100
    )

    print(f"Fold {fold_number}:")
    print(f"  Total:     {len(fold_data):,}")
    print(f"  Stayed:    {fold_counts[False]:,}")
    print(f"  Resigned:  {fold_counts[True]:,}")
    print(f"  Resignation rate: {fold_rate:.2f}%")
    print()


    # Verify that the fold resignation rate is very close
    # to the overall development resignation rate.
    if abs(fold_rate - development_rate) > 0.05:
        raise ValueError(
            f"Fold {fold_number} resignation rate "
            "differs too much from the development rate."
        )


print("CV fold stratification verification: PASSED")

CV FOLD STRATIFICATION CHECK
---------------------------------------------
Development resignation rate: 10.01%

Fold 1:
  Total:     16,000
  Stayed:    14,399
  Resigned:  1,601
  Resignation rate: 10.01%

Fold 2:
  Total:     16,000
  Stayed:    14,399
  Resigned:  1,601
  Resignation rate: 10.01%

Fold 3:
  Total:     16,000
  Stayed:    14,398
  Resigned:  1,602
  Resignation rate: 10.01%

Fold 4:
  Total:     16,000
  Stayed:    14,398
  Resigned:  1,602
  Resignation rate: 10.01%

Fold 5:
  Total:     16,000
  Stayed:    14,398
  Resigned:  1,602
  Resignation rate: 10.01%

CV fold stratification verification: PASSED


In [10]:
# ------------------------------------------------------------
# BUILD THE SHARED SPLIT REGISTRY
# ------------------------------------------------------------

# Start with Employee_ID only.
# Employee_ID is used for record tracking and will NOT be used
# as an input feature when training the machine-learning models.
split_registry = df[["Employee_ID"]].copy()


# ------------------------------------------------------------
# ADD THE DEVELOPMENT / HOLDOUT PARTITION
# ------------------------------------------------------------

# Start by marking every record as holdout.
split_registry["partition"] = "holdout"

# Replace the partition with "development" for employees that
# belong to the official 80% development set.
split_registry.loc[
    split_registry["Employee_ID"].isin(development_ids),
    "partition"
] = "development"


# ------------------------------------------------------------
# ADD THE CROSS-VALIDATION FOLD
# ------------------------------------------------------------

# Create a mapping from Employee_ID to its assigned CV fold.
# Only development employees have CV folds.
employee_to_fold = (
    development_df
    .set_index("Employee_ID")["cv_fold"]
)


# Map the fold numbers back onto the full 100,000-record registry.
# Holdout employees will automatically receive a missing value.
split_registry["cv_fold"] = (
    split_registry["Employee_ID"]
    .map(employee_to_fold)
    .astype("Int64")
)


# ------------------------------------------------------------
# SORT FOR CONSISTENT AND EASY-TO-READ OUTPUT
# ------------------------------------------------------------

# Sort by Employee_ID so the saved registry has a stable,
# predictable order.
split_registry = (
    split_registry
    .sort_values("Employee_ID")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# DISPLAY A PREVIEW
# ------------------------------------------------------------

print("SHARED SPLIT REGISTRY PREVIEW")
print("-" * 45)

display(split_registry.head(10))

print()
print(f"Registry rows: {len(split_registry):,}")

SHARED SPLIT REGISTRY PREVIEW
---------------------------------------------


,Employee_ID,partition,cv_fold
0,1,development,3
1,2,development,3
2,3,holdout,<NA>
3,4,development,5
4,5,development,3
5,6,holdout,<NA>
6,7,development,3
7,8,development,5
8,9,development,5
9,10,development,5



Registry rows: 100,000


In [11]:
# ------------------------------------------------------------
# FINAL VALIDATION OF THE SHARED SPLIT REGISTRY
# ------------------------------------------------------------

# Check the total number of registry rows.
if len(split_registry) != len(df):
    raise ValueError(
        "Registry row count does not match the dataset row count."
    )


# Check that every Employee_ID appears exactly once.
if split_registry["Employee_ID"].isna().any():
    raise ValueError(
        "The split registry contains missing Employee_ID values."
    )

if split_registry["Employee_ID"].duplicated().any():
    raise ValueError(
        "The split registry contains duplicate Employee_ID values."
    )


# Check that only the two expected partition labels are present.
expected_partitions = {"development", "holdout"}
actual_partitions = set(split_registry["partition"].unique())

if actual_partitions != expected_partitions:
    raise ValueError(
        f"Unexpected partition labels found: {actual_partitions}"
    )


# Count records in each partition.
partition_counts = split_registry["partition"].value_counts()

development_count = partition_counts.get("development", 0)
holdout_count = partition_counts.get("holdout", 0)


# Verify the expected 80/20 partition sizes.
if development_count != 80_000:
    raise ValueError(
        f"Expected 80,000 development records, found {development_count:,}."
    )

if holdout_count != 20_000:
    raise ValueError(
        f"Expected 20,000 holdout records, found {holdout_count:,}."
    )


# Verify that every development record has a CV fold.
development_registry = split_registry[
    split_registry["partition"] == "development"
]

if development_registry["cv_fold"].isna().any():
    raise ValueError(
        "At least one development record is missing a CV fold."
    )


# Verify that holdout records do NOT have CV folds.
holdout_registry = split_registry[
    split_registry["partition"] == "holdout"
]

if holdout_registry["cv_fold"].notna().any():
    raise ValueError(
        "At least one holdout record incorrectly has a CV fold."
    )


# Verify that the development data uses exactly folds 1 to 5.
actual_folds = set(
    development_registry["cv_fold"]
    .dropna()
    .astype(int)
    .unique()
)

if actual_folds != {1, 2, 3, 4, 5}:
    raise ValueError(
        f"Unexpected CV fold values found: {actual_folds}"
    )


# Verify that each CV fold contains exactly 16,000 records.
fold_counts = (
    development_registry["cv_fold"]
    .value_counts()
    .sort_index()
)

if not (fold_counts == 16_000).all():
    raise ValueError(
        "The five CV folds do not each contain exactly 16,000 records."
    )


# Display the final validation summary.
print("FINAL SPLIT REGISTRY VALIDATION")
print("-" * 45)

print(f"Total registry rows: {len(split_registry):,}")
print(f"Development rows:    {development_count:,}")
print(f"Holdout rows:        {holdout_count:,}")
print()

for fold_number, count in fold_counts.items():
    print(f"Fold {int(fold_number)}: {count:,}")

print("\nFinal split registry validation: PASSED")

FINAL SPLIT REGISTRY VALIDATION
---------------------------------------------
Total registry rows: 100,000
Development rows:    80,000
Holdout rows:        20,000

Fold 1: 16,000
Fold 2: 16,000
Fold 3: 16,000
Fold 4: 16,000
Fold 5: 16,000

Final split registry validation: PASSED


In [12]:
# ------------------------------------------------------------
# SAVE THE OFFICIAL SHARED SPLIT REGISTRY
# ------------------------------------------------------------

# Save the validated registry as a CSV file.
# This file will be shared by all six group members.
#
# Use UTF-8 encoding and a fixed "\n" line ending so the saved
# CSV is byte-for-byte reproducible across Windows, Linux, and macOS.
split_registry.to_csv(
    split_registry_file,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)


# ------------------------------------------------------------
# VERIFY THAT THE FILE WAS SAVED CORRECTLY
# ------------------------------------------------------------

# Confirm that the file now exists.
if not split_registry_file.exists():
    raise FileNotFoundError(
        "The split registry file was not saved successfully."
    )


# Reload the saved registry using a nullable integer type for cv_fold.
# This keeps fold numbers as integers while still allowing holdout rows
# to contain missing fold assignments.
saved_registry = pd.read_csv(
    split_registry_file,
    dtype={"cv_fold": "Int64"},
)


# Verify the saved row count.
if len(saved_registry) != 100_000:
    raise ValueError(
        f"Saved registry should contain 100,000 rows, "
        f"but contains {len(saved_registry):,}."
    )


# Verify the saved column names.
expected_registry_columns = [
    "Employee_ID",
    "partition",
    "cv_fold",
]

if saved_registry.columns.tolist() != expected_registry_columns:
    raise ValueError(
        "Saved registry columns do not match the expected structure."
    )


# Verify that every development record still has a CV fold
# after the CSV has been saved and reloaded.
saved_development = saved_registry[
    saved_registry["partition"] == "development"
]

if saved_development["cv_fold"].isna().any():
    raise ValueError(
        "Reloaded registry contains a development record "
        "without a CV fold."
    )


# Verify that holdout records still have no CV fold
# after serialization and reload.
saved_holdout = saved_registry[
    saved_registry["partition"] == "holdout"
]

if saved_holdout["cv_fold"].notna().any():
    raise ValueError(
        "Reloaded registry contains a holdout record "
        "with a CV fold assignment."
    )


# Display the final save summary.
print("SPLIT REGISTRY SAVED SUCCESSFULLY")
print("-" * 45)

print("File:", split_registry_file)
print(f"Rows saved: {len(saved_registry):,}")
print("Columns:", saved_registry.columns.tolist())

print("\nOfficial split registry creation: COMPLETE")

SPLIT REGISTRY SAVED SUCCESSFULLY
---------------------------------------------
File: D:\Desktop\Employee_Resignation_Prediction\data\processed\split_registry.csv
Rows saved: 100,000
Columns: ['Employee_ID', 'partition', 'cv_fold']

Official split registry creation: COMPLETE


In [13]:
# ------------------------------------------------------------
# GENERATE AND VERIFY CHECKSUM FOR THE OFFICIAL SPLIT REGISTRY
# ------------------------------------------------------------

# Official frozen SHA-256 checksum for the standardized split registry.
EXPECTED_REGISTRY_SHA256 = (
    "87edc10ac38f2435c49beddbf412f9de293e3e0622558b27ffbdf31fadd7ea41"
)


# Create a SHA-256 calculator.
registry_sha256 = hashlib.sha256()


# Read the saved registry file in chunks.
# This generates a unique fingerprint for the exact file.
with open(split_registry_file, "rb") as file:
    for chunk in iter(lambda: file.read(1024 * 1024), b""):
        registry_sha256.update(chunk)


# Store the final SHA-256 checksum.
split_registry_checksum = registry_sha256.hexdigest()


# Display the checksum.
print("OFFICIAL SPLIT REGISTRY CHECKSUM")
print("-" * 45)

print("File:            ", split_registry_file)
print("Expected SHA-256:", EXPECTED_REGISTRY_SHA256)
print("Actual SHA-256:  ", split_registry_checksum)


# Verify that the generated registry matches the frozen project checksum.
if split_registry_checksum != EXPECTED_REGISTRY_SHA256:
    raise ValueError(
        "Split registry checksum does not match the frozen project checksum."
    )

print("\nOfficial split registry checksum verification: PASSED")
print("All six model workflows should use this exact registry.")


OFFICIAL SPLIT REGISTRY CHECKSUM
---------------------------------------------
File:             D:\Desktop\Employee_Resignation_Prediction\data\processed\split_registry.csv
Expected SHA-256: 87edc10ac38f2435c49beddbf412f9de293e3e0622558b27ffbdf31fadd7ea41
Actual SHA-256:   87edc10ac38f2435c49beddbf412f9de293e3e0622558b27ffbdf31fadd7ea41

Official split registry checksum verification: PASSED
All six model workflows should use this exact registry.
